<a href="https://colab.research.google.com/github/sebabecerra/macro-plots-public/blob/main/simple_ytd_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple YTD Template

Notebook simple para descargar una serie desde Yahoo Finance, limpiarla, calcular el YTD por ano y graficar el ano actual contra los anos historicos.

[Abrir en Colab](https://colab.research.google.com/github/sebabecerra/macro-plots/blob/main/notebooks/simple_ytd_template.ipynb)

Orden del notebook:
1. Instalar librerias
2. Importar librerias
3. Definir ticker y metadatos
4. Descargar y limpiar datos
5. Construir YTD por ano
6. Calcular resumen del ano actual
7. Graficar
8. Opcional: usar un archivo propio con columnas `fecha` y `valor`


In [1]:
pip install -q pandas plotly yfinance openpyxl


## 1. Librerias y funciones

Esta celda importa las librerias y define funciones reutilizables para no repetir la misma logica varias veces.

Funciones principales:
- `clean_series(...)`: estandariza una serie con fecha y valor
- `build_ytd(...)`: construye el YTD ano por ano
- `summarize_ytd(...)`: calcula el resumen del ano actual
- `plot_ytd_chart(...)`: grafica adaptando el eje segun la frecuencia de la serie


In [9]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf

pio.renderers.default = "colab"

ACCENT = "#ffd166"
HISTORICAL = "rgba(180,180,180,0.26)"


def clean_series(frame: pd.DataFrame, date_col: str, value_col: str, keep_positive_only: bool = True) -> pd.DataFrame:
    data = (
        frame[[date_col, value_col]]
        .rename(columns={date_col: "date", value_col: "value"})
        .assign(
            date=lambda df: pd.to_datetime(df["date"], errors="coerce"),
            value=lambda df: pd.to_numeric(df["value"], errors="coerce"),
        )
        .dropna(subset=["date", "value"])
    )

    if keep_positive_only:
        data = data.loc[data["value"] > 0]
    else:
        data = data.loc[data["value"] != 0]

    data = (
        data.sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    if data.empty:
        raise ValueError("La serie no tiene observaciones validas despues de la limpieza.")

    return data


def build_ytd(data: pd.DataFrame) -> pd.DataFrame:
    ytd = pd.concat(
        [
            year_df.assign(
                year=year,
                period=range(1, len(year_df) + 1),
                change_pct=((year_df["value"] / year_df.iloc[0]["value"]) - 1.0) * 100.0,
            )
            for year, year_df in data.assign(year=data["date"].dt.year).groupby("year", sort=True)
            if not year_df.empty and year_df.iloc[0]["value"] > 0
        ],
        ignore_index=True,
    )
    if ytd.empty:
        raise ValueError("No fue posible construir el YTD para esta serie.")
    ytd["is_current"] = ytd["year"] == ytd["year"].max()
    return ytd


def summarize_ytd(ytd: pd.DataFrame) -> dict:
    current = ytd[ytd["is_current"]].copy()
    return {
        "current_year": int(current["year"].iloc[0]),
        "as_of": current["date"].max().strftime("%Y-%m-%d"),
        "current_ytd_pct": round(float(current["change_pct"].iloc[-1]), 2),
        "min_ytd_pct": round(float(current["change_pct"].min()), 2),
        "max_ytd_pct": round(float(current["change_pct"].max()), 2),
        "start_year": int(ytd["year"].min()),
        "end_year": int(ytd["year"].max()),
    }


def infer_period_label(data: pd.DataFrame) -> str:
    gaps = data["date"].sort_values().diff().dropna().dt.days
    if gaps.empty:
        return "Observation Number"
    median_gap = gaps.median()
    if median_gap >= 27:
        return "Month of Year"
    if median_gap >= 6:
        return "Week of Year"
    return "Trading Days"


def plot_ytd_chart(ytd: pd.DataFrame, title: str, unit: str, period_label: str) -> go.Figure:
    current_year = int(ytd["year"].max())
    fig = go.Figure()

    for year, year_df in ytd.groupby("year", sort=True):
        is_current = year == current_year
        fig.add_trace(go.Scatter(
            x=year_df["period"],
            y=year_df["change_pct"],
            mode="lines",
            name=str(year),
            showlegend=is_current,
            line=dict(color=ACCENT if is_current else HISTORICAL, width=3 if is_current else 1),
            customdata=year_df[["value", "date"]].assign(date=lambda df: df["date"].dt.strftime("%Y-%m-%d")).to_numpy(),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                f"{period_label} %{{x}}<br>"
                "YTD %{y:.2f}%<br>"
                "Value %{customdata[0]:,.2f}<br>"
                "Date %{customdata[1]}"
                "<extra></extra>"
            ),
        ))

    fig.update_layout(
        width=1200,
        height=720,
        paper_bgcolor="#000000",
        plot_bgcolor="#050505",
        font=dict(color="#f2f2f2", family="Arial, sans-serif", size=13),
        xaxis=dict(title=period_label, showgrid=False, zeroline=False, tickfont=dict(size=12), title_font=dict(size=15)),
        yaxis=dict(title="YTD Change (%)", ticksuffix="%", gridcolor="rgba(255,255,255,0.10)", tickfont=dict(size=12), title_font=dict(size=15)),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, font=dict(size=12)),
        hovermode="closest",
        hoverdistance=24,
        spikedistance=-1,
        margin=dict(l=90, r=50, t=130, b=80),
        annotations=[
            dict(text=f"<b>{title}</b>", x=0, y=1.14, xref="paper", yref="paper", xanchor="left", yanchor="bottom", showarrow=False, font=dict(color="#f2f2f2", size=24)),
            dict(text=unit, x=0, y=1.08, xref="paper", yref="paper", xanchor="left", yanchor="bottom", showarrow=False, font=dict(color="rgba(220,220,220,0.75)", size=15)),
        ],
    )
    return fig


## 2. Configuracion

Aqui defines que serie descargar y como quieres mostrarla en el grafico.
- `TICKER`: identificador de Yahoo Finance
- `TITLE`: titulo principal del grafico
- `UNIT`: subtitulo o unidad
- `START` y `END`: rango de fechas de descarga


In [3]:
TICKER = "CL=F"
TITLE = "WTI Crude Oil"
UNIT = "USD per barrel"
START = "1986-01-01"
END = None

## 3. Descarga y limpieza

Esta celda hace tres cosas:
1. Descarga la serie desde Yahoo Finance.
2. Selecciona una sola columna de precios (`Adj Close` si existe; si no, `Close`).
3. Limpia la tabla para dejarla lista para el calculo YTD.

Reglas de limpieza:
- convierte fecha y valor a tipos correctos
- elimina filas vacias
- deja solo valores mayores que cero
- ordena por fecha
- elimina fechas duplicadas


In [4]:
raw = yf.download(
    TICKER,
    start=START,
    end=END,
    auto_adjust=False,
    progress=False,
)

if raw.empty:
    raise ValueError(f"No data returned for {TICKER}")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

price_col = "Adj Close" if "Adj Close" in raw.columns else "Close"
data = clean_series(raw.reset_index(), date_col="Date", value_col=price_col, keep_positive_only=True)
period_label = infer_period_label(data)

data.head()


Price,date,value
0,2000-08-23,32.049999
1,2000-08-24,31.629999
2,2000-08-25,32.049999
3,2000-08-28,32.869999
4,2000-08-29,32.720001


## 4. Construccion del YTD

Esta celda separa los datos por ano calendario. Para cada ano:
- toma el primer valor valido del ano como base
- cuenta los dias de trading (`day`)
- calcula el cambio porcentual acumulado (`change_pct`)

El resultado final se guarda en `ytd`.


In [5]:
ytd = build_ytd(data)
ytd.head()


Price,date,value,year,period,change_pct,is_current
0,2000-08-23,32.049999,2000,1,0.000000,False
1,2000-08-24,31.629999,2000,2,-1.310453,False
2,2000-08-25,32.049999,2000,3,0.000000,False
3,2000-08-28,32.869999,2000,4,2.558501,False
4,2000-08-29,32.720001,2000,5,2.090490,False


## 5. Resumen del ano actual

Esta celda resume el comportamiento del ano actual:
- ano en curso
- fecha del ultimo dato
- YTD actual
- minimo YTD del ano
- maximo YTD del ano
- primer y ultimo ano de cobertura


In [6]:
summary = summarize_ytd(ytd)
summary


{'current_year': 2026,
 'as_of': '2026-03-26',
 'current_ytd_pct': 64.01,
 'min_ytd_pct': -2.32,
 'max_ytd_pct': 72.21,
 'start_year': 2000,
 'end_year': 2026}

## 6. Grafico final

Esta celda dibuja el grafico YTD:
- linea amarilla: ano actual
- lineas grises: anos historicos
- eje X: dias de trading
- eje Y: cambio porcentual acumulado YTD

El hover muestra ano, dia de trading, YTD, valor y fecha.


In [10]:
fig = plot_ytd_chart(ytd, title=TITLE, unit=UNIT, period_label=period_label)
fig.show()


## 7. Tabla final procesada

Si quieres revisar las ultimas filas de la tabla ya transformada, ejecuta esta celda.


In [11]:
ytd.tail()


Price,date,value,year,period,change_pct,is_current
6419,2026-03-20,98.320000,2026,54,71.528263,True
6420,2026-03-23,88.129997,2026,55,53.750868,True
6421,2026-03-24,92.349998,2026,56,61.113048,True
6422,2026-03-25,90.320000,2026,57,57.571529,True
6423,2026-03-26,94.010002,2026,58,64.009076,True


## 8. Ejemplo generico para cualquier dataset

Esta celda sirve para cualquier archivo propio (`csv`, `xlsx` o `json`).

Supuesto: la primera columna del archivo es la fecha y la segunda columna es el valor. No importa como se llamen originalmente.

La celda hace todo de una vez: lee el archivo, renombra las dos primeras columnas a `fecha` y `valor`, limpia la serie, detecta la frecuencia, construye el YTD, calcula el resumen y grafica.


In [16]:
FILE_PATH = "https://raw.githubusercontent.com/sebabecerra/macro-plots-public/main/ipsa.xlsx"

TITLE = "Mi Serie"
UNIT = "Nivel"

if FILE_PATH.endswith(".csv"):
    raw = pd.read_csv(FILE_PATH)
elif FILE_PATH.endswith(".xlsx"):
    raw = pd.read_excel(FILE_PATH, engine="openpyxl")
elif FILE_PATH.endswith(".json"):
    raw = pd.read_json(FILE_PATH)
else:
    raise ValueError("Formato no soportado")

source_frame = raw.iloc[:, :2].copy()
source_frame.columns = ["fecha", "valor"]

data = clean_series(source_frame, date_col="fecha", value_col="valor", keep_positive_only=True)
period_label = infer_period_label(data)

ytd = build_ytd(data)
summary = summarize_ytd(ytd)
fig = plot_ytd_chart(ytd, title=TITLE, unit=UNIT, period_label=period_label)

summary
fig.show()
ytd.tail()


/tmp/ipykernel_4554/1342706705.py:17: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,date,value,year,period,change_pct,is_current
9025,2026-03-19,10473.45,2026,55,0.429778,True
9026,2026-03-20,10277.51,2026,56,-1.449088,True
9027,2026-03-23,10227.64,2026,57,-1.927291,True
9028,2026-03-24,10206.16,2026,58,-2.133262,True
9029,2026-03-25,10409.95,2026,59,-0.179122,True
